## Kubernetes Integration

Zuerst starten wir die Microservices in Kubernetes, hier der AutoShop WebShop

In [ ]:
%%bash
kubectl create namespace ms-bkst
kubectl apply --namespace ms-bkst -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-deployment/catalog-deployment.yaml
kubectl apply --namespace ms-bkst -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-deployment/customer-deployment.yaml
kubectl apply --namespace ms-bkst -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-deployment/order-deployment.yaml
kubectl apply --namespace ms-bkst -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-deployment/webshop-deployment.yaml 
kubectl apply --namespace ms-bkst -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/catalog-service.yaml
kubectl apply --namespace ms-bkst -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/customer-service.yaml
kubectl apply --namespace ms-bkst -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/order-service.yaml
kubectl apply --namespace ms-bkst -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/webshop-service.yaml
kubectl get pod,services --namespace ms-bkst

### Kubernetes RBAC

Richten die Rechte für Backstage zum lesen der Konfiguration ein.

Dafür brauchst es eine ClusterRole mit ClusterRoleBinding


In [ ]:
%%bash
kubectl apply -f - <<'EOF'
apiVersion: v1
kind: ServiceAccount
metadata:
  name: backstage-reader
  namespace: ms-bkst
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: backstage-reader
rules:
  - apiGroups: [""]
    resources:
      - pods
      - pods/log
      - services
      - configmaps
      - events
      - resourcequotas
      - limitranges
    verbs: [get, list, watch]

  - apiGroups: ["apps"]
    resources:
      - deployments
      - replicasets
      - statefulsets
      - daemonsets
    verbs: [get, list, watch]

  - apiGroups: ["batch"]
    resources:
      - jobs
      - cronjobs
    verbs: [get, list, watch]

  - apiGroups: ["autoscaling"]
    resources:
      - horizontalpodautoscalers
    verbs: [get, list, watch]

  - apiGroups: ["networking.k8s.io"]
    resources:
      - ingresses
    verbs: [get, list, watch]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: backstage-reader
subjects:
  - kind: ServiceAccount
    name: backstage-reader
    namespace: ms-bkst
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: backstage-reader
EOF

### Token erstellen

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: Secret
metadata:
  name: backstage-reader-token
  namespace: ms-bkst
  annotations:
    kubernetes.io/service-account.name: backstage-reader
type: kubernetes.io/service-account-token
EOF

### Zugriff prüfen

In [ ]:
%%bash
kubectl auth can-i list pods --all-namespaces --as=system:serviceaccount:ms-bkst:backstage-reader

### Label suche

In [ ]:
%%bash
kubectl get all --all-namespaces -l backstage.io/kubernetes-id=shop-catalog

### Backstage-Konfiguration

In [ ]:
%%bash
export K8S_TOKEN="$(kubectl get secret backstage-reader-token --namespace ms-bkst -o jsonpath='{.data.token}' |   base64 --decode)"
export K8s_URL=$(kubectl config view --minify -o jsonpath='{.clusters[0].cluster.server}{"\n"}')

cat <<EOF | tee ~/mybackstage/app-config.kubernetes.yaml 
kubernetes:
  serviceLocatorMethod:
    type: multiTenant

  clusterLocatorMethods:
    - type: config
      clusters:
        - name: local
          url: ${K8s_URL}
          authProvider: serviceAccount
          serviceAccountToken: ${K8S_TOKEN}
          skipTLSVerify: true
          skipMetricsLookup: true
EOF


### Konfiguration prüfen

Mit `yarn backstage-cli config:print` wird die zusammengeführte und aufgelöste Backstage-Konfiguration ausgegeben, ohne die Anwendung zu starten.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Kubernetes"
export BACKSTAGE_PORT="3002"

source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn backstage-cli config:print --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.production.yaml --config ~/mybackstage/app-config.kubernetes.yaml

### Backstage starten

Backstage wird mit den gewünschten Konfigurationsdateien gestartet.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Produktion"
export BACKSTAGE_PORT="3002"

echo "http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn start --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.production.yaml --config ~/mybackstage/app-config.kubernetes.yaml

### Aufräumen

In [ ]:
%%bash
kubectl delete ns ms-bkst